# Klasifikasi Kepuasan Pelanggan Produk GEOFFMAX
## Berdasarkan Ulasan E-Commerce Shopee — Algoritma Naïve Bayes (CRISP-DM)

| | |
|---|---|
| **Objek** | Produk Sepatu GEOFFMAX di Shopee |
| **Algoritma** | Multinomial Naïve Bayes |
| **Metodologi** | CRISP-DM |
| **Balancing** | SMOTE (Synthetic Minority Oversampling Technique) |
| **Validasi** | 10-Fold Stratified Cross Validation |
| **Kelas** | 2 Kelas: Tidak Puas · Puas |
| **Data** | 400 ulasan produk sepatu GEOFFMAX di Shopee |
| **Kolom** | `ulasan` (teks) · `rating` (1–5) |

---

### Kategorisasi Label Y dari Rating:
| Rating | Label | Kode |
|---|---|---|
| ⭐ 1–2 | Tidak Puas | 0 |
| ⭐⭐⭐ 3–4–5 | Puas | 1 |


## Sel 0 — Install & Import Library
> Jalankan sel ini pertama kali. Tunggu sampai muncul ✅


In [ ]:
# Install library NLP Bahasa Indonesia dan balancing
!pip install PySastrawi imbalanced-learn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib
import warnings
warnings.filterwarnings('ignore')

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from imblearn.over_sampling import SMOTE

from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score, roc_auc_score, roc_curve)

np.random.seed(42)
print("✅ Semua library berhasil diimport.")
print("   PySastrawi    — preprocessing teks Bahasa Indonesia")
print("   imbalanced-learn — SMOTE untuk balancing data")
print("   scikit-learn  — Naive Bayes + evaluasi model")


## Sel 1 — Upload Data Asli GEOFFMAX (CRISP-DM: Data Understanding)
> Klik **Choose Files** lalu pilih file `data_ulasan_dublin_bersih__1_.csv`


In [ ]:
from google.colab import files

print("Silakan upload file CSV data ulasan GEOFFMAX...")
uploaded = files.upload()
nama_file = list(uploaded.keys())[0]

df_raw = pd.read_csv(nama_file)

print()
print(f"✅ File berhasil dimuat: {nama_file}")
print(f"   Jumlah baris  : {len(df_raw)}")
print(f"   Kolom         : {list(df_raw.columns)}")
print(f"   Missing values: {df_raw.isnull().sum().sum()}")
print()
df_raw.head(10)


## Sel 2 — Data Understanding (CRISP-DM Tahap 2)
Eksplorasi distribusi data sebelum diolah.


In [ ]:
# Buat label Y dari rating (2 kelas)
def buat_label(r):
    if r <= 2:  return 0   # Tidak Puas
    else:       return 1   # Puas (rating 3,4,5)

LABEL_MAP = {0: 'Tidak Puas', 1: 'Puas'}

df_raw['Y_Kode']  = df_raw['rating'].apply(buat_label)
df_raw['Y_Label'] = df_raw['Y_Kode'].map(LABEL_MAP)

# Distribusi rating
print("=== DISTRIBUSI RATING BINTANG ===")
rating_dist = df_raw['rating'].value_counts().sort_index()
for r, cnt in rating_dist.items():
    bintang = '⭐' * r
    print(f"  Rating {r} {bintang}: {cnt} ulasan")
print()

# Distribusi kelas Y
print("=== DISTRIBUSI KELAS Y (2 KELAS) ===")
dist = df_raw['Y_Label'].value_counts()
pct  = df_raw['Y_Label'].value_counts(normalize=True)*100
print(pd.DataFrame({'Jumlah': dist, 'Persentase (%)': pct.round(2)}))
print()
print("Catatan: Rating 1-2 = Tidak Puas | Rating 3-4-5 = Puas")


In [ ]:
# Visualisasi distribusi data
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Grafik 1: Distribusi rating
rating_cnt = df_raw['rating'].value_counts().sort_index()
warna_rating = ['#E74C3C','#E74C3C','#27AE60','#27AE60','#27AE60']
bars = axes[0].bar(rating_cnt.index, rating_cnt.values,
                   color=warna_rating, edgecolor='white', width=0.6)
for bar, v in zip(bars, rating_cnt.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.5,
                 str(v), ha='center', fontweight='bold', fontsize=11)
axes[0].set_title('Distribusi Rating Bintang', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rating'); axes[0].set_ylabel('Jumlah Ulasan')
axes[0].set_xticks([1,2,3,4,5])
axes[0].axvline(x=2.5, color='gray', linestyle='--', alpha=0.5)
axes[0].text(1.5, max(rating_cnt.values)*0.9, 'Tidak
Puas',
             ha='center', color='#E74C3C', fontsize=10, fontweight='bold')
axes[0].text(4, max(rating_cnt.values)*0.9, 'Puas',
             ha='center', color='#27AE60', fontsize=10, fontweight='bold')

# Grafik 2: Distribusi kelas Y
labels_y  = ['Tidak Puas', 'Puas']
warna_y   = ['#E74C3C', '#27AE60']
counts_y  = [df_raw[df_raw['Y_Label']==l].shape[0] for l in labels_y]
bars2 = axes[1].bar(labels_y, counts_y, color=warna_y,
                    edgecolor='white', width=0.5)
for bar, v in zip(bars2, counts_y):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.5,
                 str(v), ha='center', fontweight='bold', fontsize=13)
axes[1].set_title('Distribusi Kelas Kepuasan (Y)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Jumlah Ulasan')

plt.tight_layout()
plt.savefig('distribusi_data.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik tersimpan: distribusi_data.png")
print("   Screenshot untuk Bab IV — Data Understanding")


## Sel 3 — Data Preparation (CRISP-DM Tahap 3)

Tahapan preprocessing teks:
1. **Case folding** — semua huruf kecil
2. **Cleaning** — hapus angka, tanda baca, karakter khusus
3. **Stopword removal** — hapus kata tidak bermakna (Sastrawi + slang)
4. **Stemming** — ubah kata ke bentuk dasar (Sastrawi)
5. **TF-IDF Vectorization** — ubah teks → vektor numerik
6. **SMOTE** — seimbangkan jumlah data tiap kelas

> ⚠️ Proses ini membutuhkan waktu **1-2 menit**. Tunggu hingga muncul ✅


In [ ]:
# Inisialisasi Sastrawi
factory_sw = StopWordRemoverFactory()
stopwords  = set(factory_sw.get_stop_words())

# Tambah stopwords slang e-commerce
slang_sw = {'yg','dgn','utk','krn','sdh','blm','ga','gak','nggak',
            'udah','udh','bgt','aja','sih','deh','nih','loh','dong',
            'emg','emang','tp','sy','gw','lo','lu','ok','oke','ya',
            'yah','lah','kah','nya','si','wkwk','haha','nih','sih'}
all_stopwords = stopwords | slang_sw

factory_st = StemmerFactory()
stemmer    = factory_st.create_stemmer()

def preprocess(teks):
    # 1. Case folding
    teks = str(teks).lower()
    # 2. Cleaning
    teks = re.sub(r'[^a-z\s]', ' ', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    # 3. Stopword removal + 4. Stemming
    kata = [stemmer.stem(w) for w in teks.split()
            if w not in all_stopwords and len(w) > 1]
    return ' '.join(kata)

print("Menjalankan preprocessing teks... (1-2 menit)")
df_raw['ulasan_bersih'] = df_raw['ulasan'].apply(preprocess)
print(f"✅ Preprocessing selesai! {len(df_raw)} ulasan diproses.")
print()
print("Contoh hasil preprocessing:")
print("-" * 65)
contoh = df_raw[['ulasan','ulasan_bersih','Y_Label']].head(6)
for _, row in contoh.iterrows():
    print(f"ASLI    : {str(row['ulasan'])[:65]}")
    print(f"BERSIH  : {row['ulasan_bersih']}")
    print(f"LABEL   : {row['Y_Label']}")
    print()


In [ ]:
# TF-IDF Vectorization dengan parameter terbaik hasil tuning
# (alpha=0.01, max_features=1000, ngram=(1,1))
tfidf_final = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 1),
    min_df=1,
    sublinear_tf=True
)

X_raw = tfidf_final.fit_transform(df_raw['ulasan_bersih'])
y_raw = df_raw['Y_Kode'].values

print(f"✅ TF-IDF selesai!")
print(f"   Shape matriks      : {X_raw.shape}")
print(f"   Jumlah fitur (kata): {X_raw.shape[1]}")
print(f"   Jumlah dokumen     : {X_raw.shape[0]}")
print()

# SMOTE untuk menyeimbangkan kelas
smote = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X_raw.toarray(), y_raw)

print(f"✅ SMOTE selesai!")
print(f"   Sebelum: Tidak Puas={sum(y_raw==0)}, Puas={sum(y_raw==1)}")
print(f"   Sesudah: Tidak Puas={sum(y_sm==0)}, Puas={sum(y_sm==1)}")


In [ ]:
# Visualisasi Top 20 kata TF-IDF
fitur_names = tfidf_final.get_feature_names_out()
mean_tfidf  = np.asarray(X_raw.mean(axis=0)).flatten()
top20_idx   = mean_tfidf.argsort()[-20:][::-1]
top20_kata  = [fitur_names[i] for i in top20_idx]
top20_skor  = [mean_tfidf[i]  for i in top20_idx]

fig, ax = plt.subplots(figsize=(9,6))
ax.barh(top20_kata[::-1], top20_skor[::-1], color='#2E75B6')
ax.set_xlabel('Rata-rata Bobot TF-IDF')
ax.set_title('Top 20 Kata dengan Bobot TF-IDF Tertinggi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('top20_tfidf.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik tersimpan: top20_tfidf.png")
print("   Screenshot untuk Bab IV — Data Preparation")


## Sel 4 — Modeling: Multinomial Naïve Bayes (CRISP-DM Tahap 4)

**Rumus Naïve Bayes:**
$$P(kelas | teks) = \frac{P(teks | kelas) \cdot P(kelas)}{P(teks)}$$

**Parameter model terbaik hasil tuning:**
- `alpha = 0.01` (Laplace smoothing)
- `max_features = 1000`
- `ngram_range = (1,1)` unigram
- Balancing: **SMOTE**
- Validasi: **10-Fold Stratified Cross Validation**

> Proses sekitar **10-30 detik**


In [ ]:
# Inisialisasi model Naive Bayes dengan parameter terbaik
model_final = MultinomialNB(alpha=0.01)
skf         = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print("Menjalankan 10-Fold Cross Validation...")
print("(data SMOTE: 528 sampel — 264 Tidak Puas + 264 Puas)")
print()

# Prediksi cross-validation
y_pred  = cross_val_predict(model_final, X_sm, y_sm, cv=skf)
y_proba = cross_val_predict(model_final, X_sm, y_sm,
                             cv=skf, method='predict_proba')

# Latih model final dengan seluruh data SMOTE
model_final.fit(X_sm, y_sm)

print("✅ Pemodelan selesai!")
print(f"   Total data (setelah SMOTE)  : {len(y_sm)}")
print(f"   Total data diprediksi       : {len(y_pred)}")
print()
print("Distribusi hasil prediksi:")
for kode, label in {0:'Tidak Puas', 1:'Puas'}.items():
    jumlah = sum(y_pred == kode)
    print(f"  {label:12s}: {jumlah}")


## Sel 5 — Evaluasi Model (CRISP-DM Tahap 5)
> Screenshot semua output di bagian ini untuk **Bab IV skripsi**


In [ ]:
LABELS_TEXT  = ['Tidak Puas', 'Puas']
LABELS_ORDER = [0, 1]

# Akurasi & AUC
acc = accuracy_score(y_sm, y_pred)
auc = roc_auc_score(y_sm, y_proba[:,1])

print("╔══════════════════════════════════════════╗")
print("║       HASIL EVALUASI MODEL FINAL         ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Akurasi          : {acc*100:.2f}%               ║")
print(f"║  AUC              : {auc:.3f}                  ║")
print(f"║  Algoritma        : Multinomial Naïve Bayes  ║")
print(f"║  Balancing        : SMOTE                    ║")
print(f"║  Validasi         : 10-Fold CV               ║")
print("╚══════════════════════════════════════════╝")
print()

# Confusion Matrix
cm = confusion_matrix(y_sm, y_pred, labels=LABELS_ORDER)
cm_df = pd.DataFrame(cm,
    index  =[f'True: {l}' for l in LABELS_TEXT],
    columns=[f'Pred: {l}' for l in LABELS_TEXT])
print("Confusion Matrix:")
print(cm_df)
print()

# Classification Report
report = classification_report(y_sm, y_pred,
             target_names=LABELS_TEXT, output_dict=True)
print(classification_report(y_sm, y_pred, target_names=LABELS_TEXT))


In [ ]:
# Tabel ringkasan metrik untuk Bab IV
ringkasan = pd.DataFrame({
    'Kelas'     : LABELS_TEXT,
    'Precision' : [report[l]['precision'] for l in LABELS_TEXT],
    'Recall'    : [report[l]['recall']    for l in LABELS_TEXT],
    'F1-Score'  : [report[l]['f1-score']  for l in LABELS_TEXT],
    'Support'   : [int(report[l]['support']) for l in LABELS_TEXT],
})

print("Tabel Metrik Evaluasi per Kelas (untuk Bab IV skripsi):")
display(ringkasan.round(4))
print()
print(f"Akurasi Keseluruhan : {acc*100:.2f}%")
print(f"AUC                 : {auc:.3f}")
print(f"Macro avg F1        : {report['macro avg']['f1-score']:.4f}")


In [ ]:
# Visualisasi: Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS_TEXT, yticklabels=LABELS_TEXT,
            ax=axes[0], linewidths=0.5, annot_kws={'size':16})
axes[0].set_title('Confusion Matrix\nMultinomial Naïve Bayes + SMOTE',
                   fontsize=13, fontweight='bold')
axes[0].set_xlabel('Prediksi', fontsize=11)
axes[0].set_ylabel('Aktual', fontsize=11)

# ROC Curve
fpr, tpr, _ = roc_curve(y_sm, y_proba[:,1])
axes[1].plot(fpr, tpr, color='#2E75B6', lw=2.5,
             label=f'Naïve Bayes (AUC = {auc:.3f})')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#2E75B6')
axes[1].plot([0,1],[0,1],'--', color='gray', lw=1.5, label='Random Guess')
axes[1].set_title('ROC Curve — One-vs-Rest', fontsize=13, fontweight='bold')
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].legend(fontsize=11)
axes[1].set_xlim([0,1]); axes[1].set_ylim([0,1.02])

plt.tight_layout()
plt.savefig('evaluasi_model.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik tersimpan: evaluasi_model.png")
print("   Screenshot untuk Bab IV — Evaluasi Model")


In [ ]:
# Visualisasi: Top 10 Kata per Kelas
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
warna_kelas = ['#E74C3C', '#27AE60']

for cls_idx, (cls_name, color) in enumerate(zip(LABELS_TEXT, warna_kelas)):
    log_prob   = model_final.feature_log_prob_[cls_idx]
    top10_idx  = log_prob.argsort()[-10:][::-1]
    top10_kata = [fitur_names[i] for i in top10_idx]
    top10_skor = [log_prob[i]    for i in top10_idx]

    axes[cls_idx].barh(top10_kata[::-1], top10_skor[::-1], color=color)
    axes[cls_idx].set_title(f'Top 10 Kata — {cls_name}', fontsize=12, fontweight='bold')
    axes[cls_idx].set_xlabel('Log Probability')

plt.suptitle('Kata Paling Berpengaruh per Kelas Kepuasan',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('kata_per_kelas.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik tersimpan: kata_per_kelas.png")
print("   Screenshot untuk Bab IV — Pembahasan")


## Sel 6 — Simpan & Download Model (untuk Dashboard Streamlit Cloud)

> Dua file akan otomatis terdownload ke folder **Downloads** komputer Anda.
> Jika browser meminta izin download — klik **Izinkan / Allow**.


In [ ]:
from google.colab import files

# Simpan model dan vectorizer
joblib.dump(model_final, 'model_geoffmax.pkl')
joblib.dump(tfidf_final, 'tfidf_geoffmax.pkl')

print("✅ File berhasil disimpan di Colab storage:")
print("   model_geoffmax.pkl  — Multinomial Naïve Bayes terlatih")
print("   tfidf_geoffmax.pkl  — TF-IDF vectorizer (1000 fitur)")
print()
print("Mengunduh ke komputer Anda...")

files.download('model_geoffmax.pkl')
files.download('tfidf_geoffmax.pkl')

print()
print("╔══════════════════════════════════════════════════╗")
print("║  NOTEBOOK SELESAI ✅                             ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Akurasi  : {acc*100:.2f}%                          ║")
print(f"║  AUC      : {auc:.3f}                            ║")
print("║  Kelas    : 2 (Tidak Puas / Puas)               ║")
print("║  Balancing: SMOTE                               ║")
print("╠══════════════════════════════════════════════════╣")
print("║  File yang didownload:                          ║")
print("║  ✅ model_geoffmax.pkl                          ║")
print("║  ✅ tfidf_geoffmax.pkl                          ║")
print("╠══════════════════════════════════════════════════╣")
print("║  Langkah selanjutnya:                           ║")
print("║  1. Cek folder Downloads komputer               ║")
print("║  2. Upload 4 file ke GitHub                     ║")
print("║  3. Deploy ke share.streamlit.io                ║")
print("╚══════════════════════════════════════════════════╝")


## Sel 7 — Demo Prediksi Ulasan Baru (Opsional)
> Uji model dengan ulasan baru. Bisa diganti dengan ulasan apapun.


In [ ]:
def prediksi_ulasan(teks_baru):
    """Prediksi kepuasan dari teks ulasan baru."""
    # Preprocessing
    teks_bersih = preprocess(teks_baru)
    # Vectorize
    inp         = tfidf_final.transform([teks_bersih])
    # Prediksi
    pred_kelas  = model_final.predict(inp)[0]
    pred_proba  = model_final.predict_proba(inp)[0]

    emoji = {0: '😞', 1: '😊'}
    label = {0: 'Tidak Puas', 1: 'Puas'}

    print(f"{'='*55}")
    print(f"Ulasan   : {teks_baru}")
    print(f"Bersih   : {teks_bersih}")
    print(f"Prediksi : {emoji[pred_kelas]} {label[pred_kelas]}")
    print(f"Probabilitas:")
    for lbl, prob in zip(['Tidak Puas','Puas'], pred_proba):
        bar = '█' * int(prob * 25)
        print(f"  {lbl:12s}: {bar} {prob*100:.1f}%")
    print()

# Uji dengan berbagai jenis ulasan
prediksi_ulasan("sepatunya bagus banget kualitas oke nyaman dipakai seharian recommended")
prediksi_ulasan("kecewa banget kualitas buruk sol lepas setelah 3 hari dipakai tidak worth it")
prediksi_ulasan("lumayan sih untuk harga segini tidak jelek tapi tidak bagus juga biasa aja")
prediksi_ulasan("pengiriman cepat barang sesuai foto bahan kulit asli mantap puas banget")

# ── Coba ulasan Anda sendiri di sini ──
# prediksi_ulasan("ketik ulasan Anda di sini")
